In [13]:
import pandas as pd

df = pd.read_csv("data/raw/anand_vihar.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(70177, 29)
['Station ID', 'State', 'City', 'Station Name', 'Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)', 'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)', 'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)']


,Station ID,State,City,Station Name,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,site_301,Delhi,Delhi,"Anand Vihar, Delhi - DPCC",2024-01-01T00:00:00.000000+0000,158.0,265.0,28.4,45.9,47.3,...,NaN,11.5,80.0,0.3,44.0,NaN,0.0,5.0,742.0,NaN
1,site_301,Delhi,Delhi,"Anand Vihar, Delhi - DPCC",2024-01-01T00:15:00.000000+0000,158.0,265.0,22.9,38.5,39.1,...,NaN,11.5,81.0,0.3,43.0,NaN,0.0,5.0,742.0,NaN
2,site_301,Delhi,Delhi,"Anand Vihar, Delhi - DPCC",2024-01-01T00:30:00.000000+0000,158.0,265.0,24.4,45.2,43.9,...,NaN,11.5,80.0,0.3,50.0,NaN,0.0,5.0,742.0,NaN
3,site_301,Delhi,Delhi,"Anand Vihar, Delhi - DPCC",2024-01-01T00:45:00.000000+0000,169.0,254.0,24.8,49.6,46.6,...,NaN,11.5,80.0,0.3,78.0,NaN,0.0,5.0,742.0,NaN
4,site_301,Delhi,Delhi,"Anand Vihar, Delhi - DPCC",2024-01-01T01:00:00.000000+0000,169.0,254.0,33.4,49.5,53.5,...,NaN,11.5,80.0,0.3,77.0,NaN,0.0,5.0,742.0,NaN


In [14]:
print(df.columns.tolist())
print(df.dtypes)
print(df["Timestamp"].min(), "to", df["Timestamp"].max())
print((df.isna().mean() * 100).round(1).sort_values(ascending=False))

['Station ID', 'State', 'City', 'Station Name', 'Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)', 'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)', 'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)']
Station ID                 str
State                      str
City                       str
Station Name               str
Timestamp                  str
PM2.5 (µg/m³)          float64
PM10 (µg/m³)           float64
NO (µg/m³)             float64
NO2 (µg/m³)            float64
NOx (ppb)              float64
NH3 (µg/m³)            float64
SO2 (µg/m³)            float64
CO (mg/m³)             float64
Ozone (µg/m³)          float64
Benzene (µg/m³)        float64
Toluene (µg/m³)        float64
Xylene (µg/m³)         float64
O Xylene (µg/m³)       float64
Eth-Benzene (µ

In [15]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

is_missing = df["PM2.5 (µg/m³)"].isna()
run_id = (is_missing != is_missing.shift()).cumsum()
gap_lengths = df[is_missing].groupby(run_id).size()

print("Number of separate missing gaps:", len(gap_lengths))
print("\nGap length distribution (in 15-min intervals, so 4 = 1 hour):")
print(gap_lengths.describe())
print("\nLongest 10 gaps:")
print(gap_lengths.sort_values(ascending=False).head(10))

Number of separate missing gaps: 1569

Gap length distribution (in 15-min intervals, so 4 = 1 hour):
count    1569.000000
mean        6.772467
std        34.223863
min         1.000000
25%         1.000000
50%         1.000000
75%         4.000000
max       941.000000
dtype: float64

Longest 10 gaps:
PM2.5 (µg/m³)
144     941
488     481
1320    425
238     362
254     348
2304    309
1532    165
2440    124
610     122
456     121
dtype: int64


In [16]:
cols_to_drop = ["VWS (m/s)", "O Xylene (µg/m³)", "Eth-Benzene (µg/m³)",
                 "MP-Xylene (µg/m³)", "Xylene (µg/m³)", "RF (mm)", "TOT-RF (mm)"]
df_clean = df.drop(columns=cols_to_drop)

keep_cols = ["Timestamp", "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO2 (µg/m³)",
             "SO2 (µg/m³)", "CO (mg/m³)", "Ozone (µg/m³)",
             "AT (°C)", "RH (%)", "WS (m/s)", "WD (deg)"]
df_clean = df_clean[keep_cols]

df_clean = df_clean.rename(columns={
    "PM2.5 (µg/m³)": "pm25", "PM10 (µg/m³)": "pm10", "NO2 (µg/m³)": "no2",
    "SO2 (µg/m³)": "so2", "CO (mg/m³)": "co", "Ozone (µg/m³)": "ozone",
    "AT (°C)": "temp", "RH (%)": "humidity",
    "WS (m/s)": "wind_speed", "WD (deg)": "wind_dir",
})

df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"])
df_clean = df_clean.sort_values("Timestamp").reset_index(drop=True)

before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["Timestamp"], keep="first")
print(f"Duplicate timestamps removed: {before - len(df_clean)}")

outlier_caps = {
    "pm25": 600,
    "pm10": 800,
    "no2": 400,
    "so2": 400,
    "co": 34,
    "ozone": 400,
}

for col, cap in outlier_caps.items():
    n_outliers = (df_clean[col] > cap).sum()
    print(f"{col}: {n_outliers} values above {cap}, set to NaN")
    df_clean.loc[df_clean[col] > cap, col] = pd.NA


numeric_cols = ["pm25", "pm10", "no2", "so2", "co", "ozone", "temp", "humidity", "wind_speed", "wind_dir"]
for col in numeric_cols:
    df_clean[col] = df_clean[col].interpolate(method="linear", limit=4)

df_clean = df_clean.set_index("Timestamp")
print(df_clean.shape)
df_clean.head(10)

Duplicate timestamps removed: 0
pm25: 385 values above 600, set to NaN
pm10: 1315 values above 800, set to NaN
no2: 172 values above 400, set to NaN
so2: 0 values above 400, set to NaN
co: 0 values above 34, set to NaN
ozone: 20 values above 400, set to NaN
(70177, 10)


,pm25,pm10,no2,so2,co,ozone,temp,humidity,wind_speed,wind_dir
Timestamp,,,,,,,,,,
2024-01-01 00:00:00+00:00,158.0,265.0,45.9,4.2,1.57,20.3,11.5,80.0,0.3,44.0
2024-01-01 00:15:00+00:00,158.0,265.0,38.5,4.0,1.63,20.2,11.5,81.0,0.3,43.0
2024-01-01 00:30:00+00:00,158.0,265.0,45.2,4.0,1.58,20.3,11.5,80.0,0.3,50.0
2024-01-01 00:45:00+00:00,169.0,254.0,49.6,3.1,1.59,19.9,11.5,80.0,0.3,78.0
2024-01-01 01:00:00+00:00,169.0,254.0,49.5,4.0,1.55,19.6,11.5,80.0,0.3,77.0
2024-01-01 01:15:00+00:00,169.0,254.0,46.4,5.3,1.53,19.5,11.4,80.0,0.3,2.0
2024-01-01 01:30:00+00:00,169.0,254.0,50.6,3.5,1.49,19.0,11.4,80.0,0.3,68.0
2024-01-01 01:45:00+00:00,175.0,254.0,50.6,2.9,1.48,18.8,11.4,80.0,0.3,78.0
2024-01-01 02:00:00+00:00,175.0,254.0,50.1,3.1,1.48,18.1,11.4,80.0,0.3,62.0


In [17]:
hourly = df_clean.resample("1h").mean()
print(hourly.shape)
hourly.head(10)

(17544, 10)


,pm25,pm10,no2,so2,co,ozone,temp,humidity,wind_speed,wind_dir
Timestamp,,,,,,,,,,
2024-01-01 00:00:00+00:00,160.75,262.25,44.800,3.825,1.5925,20.175,11.500,80.25,0.300,53.75
2024-01-01 01:00:00+00:00,170.50,254.00,49.275,3.925,1.5125,19.225,11.425,80.00,0.300,56.25
2024-01-01 02:00:00+00:00,172.50,243.25,45.200,2.675,1.5975,17.725,11.350,80.00,0.300,46.75
2024-01-01 03:00:00+00:00,170.00,217.75,40.050,1.000,1.5025,15.550,11.400,80.00,0.300,28.25
2024-01-01 04:00:00+00:00,183.00,241.50,39.150,1.000,1.4800,15.475,11.300,80.00,0.300,69.75
2024-01-01 05:00:00+00:00,173.25,254.75,51.550,0.810,1.5600,14.975,11.225,80.00,0.325,34.25
2024-01-01 06:00:00+00:00,162.25,265.50,38.950,1.790,1.4175,15.250,11.125,80.50,0.525,80.00
2024-01-01 07:00:00+00:00,162.50,278.25,42.675,1.825,1.3725,15.375,11.000,80.25,0.375,31.50
2024-01-01 08:00:00+00:00,160.00,286.25,42.825,3.150,1.4925,15.100,10.975,80.00,0.650,68.00


In [18]:
missing_after = (hourly.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing_after)

so2           14.0
ozone         10.9
pm10          10.8
pm25          10.0
wind_speed     9.1
co             9.0
no2            8.7
wind_dir       8.4
humidity       8.1
temp           8.1
dtype: float64


In [19]:
hourly.to_csv("data/processed/anand_vihar_hourly.csv")
print("Saved.")

Saved.


In [20]:
data = hourly.copy()

data["hour"] = data.index.hour
data["day_of_week"] = data.index.dayofweek
data["month"] = data.index.month

pm25_lags = [1, 2, 3, 6, 12, 24]
for lag in pm25_lags:
    data[f"pm25_lag{lag}"] = data["pm25"].shift(lag)

for col, lags in {"pm10": [1, 3, 6], "no2": [1, 3], "so2": [1], "co": [1], "ozone": [1]}.items():
    for lag in lags:
        data[f"{col}_lag{lag}"] = data[col].shift(lag)

data["pm25_target_6h"] = data["pm25"].shift(-6)

feature_cols = (
    [c for c in data.columns if "_lag" in c]
    + ["temp", "humidity", "wind_speed", "wind_dir", "hour", "day_of_week", "month"]
    + ["pm25_target_6h"]
)
final = data[feature_cols]

before = len(final)
final_clean = final.dropna()
print(f"Rows before: {before} | after dropping NaN: {len(final_clean)} | dropped: {before - len(final_clean)}")
print(final_clean.shape)
final_clean.head()

Rows before: 17544 | after dropping NaN: 11548 | dropped: 5996
(11548, 22)


,pm25_lag1,pm25_lag2,pm25_lag3,pm25_lag6,pm25_lag12,pm25_lag24,pm10_lag1,pm10_lag3,pm10_lag6,no2_lag1,...,co_lag1,ozone_lag1,temp,humidity,wind_speed,wind_dir,hour,day_of_week,month,pm25_target_6h
Timestamp,,,,,,,,,,,,,,,,,,,,,
2024-01-02 00:00:00+00:00,321.50,311.25,295.00,196.75,189.75,160.75,535.50,493.875,317.500,64.725,...,1.5875,14.425,10.950,76.00,0.325,89.75,0,1,1,189.75
2024-01-02 01:00:00+00:00,299.75,321.50,311.25,206.50,214.00,170.50,473.75,542.000,323.000,63.125,...,1.4925,14.600,10.775,76.25,0.300,105.75,1,1,1,150.50
2024-01-02 02:00:00+00:00,285.00,299.75,321.50,251.25,227.50,172.50,423.00,535.500,400.000,64.225,...,1.4000,14.650,10.600,77.00,0.400,68.75,2,1,1,117.00
2024-01-02 03:00:00+00:00,275.75,285.00,299.75,295.00,222.25,170.00,398.50,473.750,493.875,54.150,...,1.4675,14.725,10.600,77.00,0.300,48.00,3,1,1,120.75
2024-01-02 04:00:00+00:00,259.75,275.75,285.00,311.25,192.75,183.00,337.25,423.000,542.000,50.025,...,1.4625,14.825,10.600,76.75,0.300,63.25,4,1,1,126.75


In [21]:
final_clean.to_csv("data/processed/anand_vihar_final.csv")
print("Saved.")

Saved.
